In [1]:
import pandas as pd
import requests
from datetime import datetime

#### I. MTA Daily Ridership and Traffic

In [ ]:
import requests

# Use the correct Socrata API endpoint (resource endpoint, not views)
url = "https://data.ny.gov/resource/sayj-mze2.json"

params = {
    "$where": "date >= '2023-01-01'",
    "$order": "date ASC",
    "$limit": 100000
}
response = requests.get(url, params=params)
data = response.json()

In [58]:
# Convert to DataFrame
df = pd.DataFrame(data)
df['date'] = pd.to_datetime(df['date'])
df['count'] = pd.to_numeric(df['count'], errors='coerce')

In [59]:
df.sample(10)

,date,mode,count
8214,2025-12-12,AAR,46724.0
5267,2025-01-18,LIRR,122763.0
5897,2025-03-29,LIRR,148404.0
1282,2023-07-03,BT,863548.0
6925,2025-07-21,SIR,6941.0
2623,2024-01-10,SIR,7216.0
5664,2025-03-03,MNR,190695.0
7869,2025-11-03,MNR,220572.0
6508,2025-06-05,CRZ Entries,539708.0
7924,2025-11-09,SIR,6.0


In [60]:
df['mode'].unique()

array(['AAR', 'BT', 'Bus', 'LIRR', 'MNR', 'SIR', 'Subway', 'CBD Entries',
       'CRZ Entries'], dtype=object)

In [61]:
# exclude Bridges and Tunnels, CBD Entries, and CRZ Entries
df = df[~df['mode'].isin(["BT", "CBD Entries", "CRZ Entries"])]

In [62]:
abb_to_name = {
    "LIRR": "Long Island Rail Road",
    "AAR": "Access-A-Ride",
    "SIR": "Staten Island Railway",
    "Subway": "Subway",
    "Bus": "Bus",
    "MNR": "Metro-North Railroad"
}

In [63]:
df['mode'] = df['mode'].map(abb_to_name)

df['mode'].unique()

array(['Access-A-Ride', 'Bus', 'Long Island Rail Road',
       'Metro-North Railroad', 'Staten Island Railway', 'Subway'],
      dtype=object)

In [64]:
df = df.sort_values(['mode', 'date']).set_index('date')
df['count_ma30'] = df.groupby('mode')['count'].transform(lambda x: x.rolling('30D', min_periods=1).mean())
df = df.reset_index()

In [65]:
# order the df by mode from highest to lowest average daily ridership
mode_order = df.groupby('mode')['count'].mean().sort_values(ascending=False).index
df['mode'] = pd.Categorical(df['mode'], categories=mode_order, ordered=True)
df = df.sort_values(['mode', 'date'])

In [66]:
df.to_csv("daily_ridership.csv", index=False)

#### II. Seasonal Adjustment for Subway and Bus Ridership